# Clasificador de Vinos con KNN

## 1. Carga de datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

URL = "https://raw.githubusercontent.com/4GeeksAcademy/k-nearest-neighbors-project-tutorial/refs/heads/main/winequality-red.csv"
df = pd.read_csv(URL, sep=";")

print("Shape:", df.shape)
df.head()

In [ ]:
# Exploración básica
print(df.info())
print("\nEstadísticas:")
df.describe()

In [ ]:
# Distribución de la columna quality original
print("Distribución de quality:")
print(df["quality"].value_counts().sort_index())

# Crear columna label: 0=baja, 1=media, 2=alta
def quality_to_label(q):
    if q <= 4:
        return 0
    elif q <= 6:
        return 1
    else:
        return 2

df["label"] = df["quality"].apply(quality_to_label)
print("\nDistribución de label:")
print(df["label"].value_counts().sort_index())

## 2. Entrenamiento del modelo KNN

In [ ]:
# Separar variables independientes (X) del objetivo (y)
X = df.drop(columns=["quality", "label"])
y = df["label"]

# Dividir en entrenamiento y prueba (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Escalar los datos
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Entrenamiento: {X_train_scaled.shape[0]} muestras")
print(f"Prueba:        {X_test_scaled.shape[0]} muestras")

In [ ]:
# Entrenar con k=7 como valor inicial
k_initial = 7
knn = KNeighborsClassifier(n_neighbors=k_initial)
knn.fit(X_train_scaled, y_train)
print(f"Modelo KNN entrenado con k={k_initial}")

## 3. Evaluación del rendimiento

In [ ]:
y_pred = knn.predict(X_test_scaled)

acc = accuracy_score(y_test, y_pred)
print(f"Accuracy (k={k_initial}): {acc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Baja", "Media", "Alta"]))

In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Baja", "Media", "Alta"],
            yticklabels=["Baja", "Media", "Alta"])
plt.title(f"Matriz de Confusión (k={k_initial})")
plt.ylabel("Real")
plt.xlabel("Predicho")
plt.tight_layout()
plt.show()

## 4. Optimización de k

In [ ]:
# Probar k de 1 a 20 y guardar resultados
k_values = list(range(1, 21))
accuracies = []

for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    accuracies.append(accuracy_score(y_test, preds))

best_k = k_values[accuracies.index(max(accuracies))]
print(f"Mejor k: {best_k}  |  Accuracy: {max(accuracies):.4f}")

In [ ]:
# Gráfica accuracy vs k
plt.figure(figsize=(9, 5))
plt.plot(k_values, accuracies, marker="o", color="steelblue", linewidth=2)
plt.axvline(best_k, color="red", linestyle="--", label=f"Mejor k={best_k}")
plt.scatter([best_k], [max(accuracies)], color="red", zorder=5)
plt.title("Accuracy vs k (KNN - Wine Quality)")
plt.xlabel("Número de vecinos (k)")
plt.ylabel("Accuracy")
plt.xticks(k_values)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Bonus: Función predict_wine_quality

In [ ]:
# Reentrenar con el mejor k
best_model = KNeighborsClassifier(n_neighbors=best_k)
best_model.fit(X_train_scaled, y_train)

label_names = {0: "Baja calidad", 1: "Calidad media", 2: "Alta calidad"}
label_emoji = {0: "🍷", 1: "🍷", 2: "🍾"}

def predict_wine_quality(features):
    """
    Predice la calidad de un vino tinto.
    
    Parámetros:
        features: lista con 11 valores numéricos en el orden:
            [fixed acidity, volatile acidity, citric acid, residual sugar,
             chlorides, free sulfur dioxide, total sulfur dioxide,
             density, pH, sulphates, alcohol]
    
    Retorna:
        str con la calidad predicha
    """
    features_array = np.array(features).reshape(1, -1)
    features_scaled = scaler.transform(features_array)
    prediction = best_model.predict(features_scaled)[0]
    quality = label_names[prediction]
    emoji = label_emoji[prediction]
    return f"Este vino probablemente sea de {quality} {emoji}"


# Ejemplo de uso
result = predict_wine_quality([7.4, 0.7, 0.0, 1.9, 0.076, 11.0, 34.0, 0.9978, 3.51, 0.56, 9.4])
print(result)